# Encoder Decoder Architecture

In [2]:
import numpy as np
import torch

### Genrate the Data

In [69]:
# build the sequence of the 5 Digits
PADDING = 0
SEQUENCE_LENGTH = 5
SOS = 10
EOS = 11
BATCH_SIZE = 32
VOCAB_SIZE = 12 #1-9+SOC+EOS
EMBEDDING_SIZE = 32
HIDDEN_SIZE = 64

In [70]:
from torch.utils.data import Dataset

class Revserse_Dataset(Dataset):
    def __init__(self,size=20000):
        #collect all samples
        self.samples = []

        #genrate the samples
        for _ in range(size):
            sequence = [np.random.randint(1,9) for _ in range(SEQUENCE_LENGTH)]

            output =list(reversed(sequence))

            self.samples.append((sequence,output))

    def __len__(self):
        return len(self.samples)
    def __getitem__(self,index):
        # get the input and output
        source,target = self.samples[index]
        # convert the source to the tensor to pass to the encoder
        encoder_input = torch.tensor(source)
        # add the SOS token to the input which will be passed to the decoder
        decoder_input = torch.tensor([SOS] + target)

        # decoder output 
        decoder_output = torch.tensor(target+[EOS])

        return encoder_input,decoder_input,decoder_output

In [71]:
dataset = Revserse_Dataset()
for value in dataset:
    print(value)
    break

(tensor([3, 1, 8, 6, 7]), tensor([10,  7,  6,  8,  1,  3]), tensor([ 7,  6,  8,  1,  3, 11]))


### Dataloader

In [72]:
from torch.utils.data import DataLoader
data_loader = DataLoader(
    dataset,
    batch_size = BATCH_SIZE,
    shuffle = True
)

### Define the Encoder

In [73]:
class Encoder(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = torch.nn.Embedding(VOCAB_SIZE,EMBEDDING_SIZE)
        self.lstm = torch.nn.LSTM(
            EMBEDDING_SIZE,
            HIDDEN_SIZE,
            batch_first = True
        )
    def forward(self,x):
        x = self.embedding(x)
        output,(hidden,cell) = self.lstm(x)
        return hidden ,cell

In [74]:
class Decoder(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = torch.nn.Embedding(VOCAB_SIZE,EMBEDDING_SIZE)
        self.lstm = torch.nn.LSTM(
            EMBEDDING_SIZE,
            HIDDEN_SIZE,
            batch_first = True
        )
        self.linear = torch.nn.Linear(HIDDEN_SIZE,VOCAB_SIZE)
    def forward(self,x,hidden,cell):
        x=self.embedding(x)
        output,(hidden,cell) = self.lstm(x,(hidden,cell))
        prediction = self.linear(output)
        return prediction,output,cell
        

### DEfinE the model ClASS

In [75]:
class Sequence2Sequence(torch.nn.Module):
    def __init__(self):
        super().__init__()

        #create the encoder decoder 
        self.encoder = Encoder()
        self.decoder = Decoder()
    def forward(self,x,decoder_input):
        # pass the input(x) to the encoder first
        hidden,cell = self.encoder(x)

        #get the output genrated by the Decoder
        output,_,_ = self.decoder(decoder_input,hidden,cell)

        return output

In [76]:
# creatre the model
model = Sequence2Sequence()

### Define the hyperparametrs

In [77]:
loss_function=torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

epochs = 500

### tRaining Loop

In [78]:
for epoch in range(epochs):
    total_loss = 0

    # Get the Data Sequence by Sequence
    for source,decoder_input,decoder_target in data_loader:
        #predict the result
        prediction = model(source,decoder_input)

        # calculate the loss 
        loss = loss_function(prediction.reshape(-1,VOCAB_SIZE),decoder_target.reshape(-1))

        # Zero out the gradients
        optimizer.zero_grad()

        # cal the gradients
        loss.backward()

        # update the parameters 
        optimizer.step()

        # Update the ttotal loss
        total_loss +=  loss.item()

    if (epoch+1) % 20 == 0:
        print(f"Epoch : {epoch+1} , loss : {total_loss}")
        

Epoch : 20 , loss : 0.14251259696902707
Epoch : 40 , loss : 0.03401627140738128
Epoch : 60 , loss : 0.00022449349593500756
Epoch : 80 , loss : 0.0008492684116276905
Epoch : 100 , loss : 5.090536989340999
Epoch : 120 , loss : 0.00011142370989247752
Epoch : 140 , loss : 0.004574879509050334
Epoch : 160 , loss : 2.3604044313785266e-05
Epoch : 180 , loss : 0.0016822554229918296
Epoch : 200 , loss : 8.596725544984807e-06
Epoch : 220 , loss : 0.0008090927326236397
Epoch : 240 , loss : 4.636743695107803e-06
Epoch : 260 , loss : 0.2149190064509412
Epoch : 280 , loss : 3.711997048139182e-05
Epoch : 300 , loss : 4.861503467568085e-07
Epoch : 320 , loss : 0.00012110631440531705
Epoch : 340 , loss : 7.643053260286692e-07
Epoch : 360 , loss : 0.0017835997793156366
Epoch : 380 , loss : 1.1139853511310527e-05
Epoch : 400 , loss : 5.072603219691274e-07
Epoch : 420 , loss : 2.1103900075866724
Epoch : 440 , loss : 2.6777333065552966e-05
Epoch : 460 , loss : 6.531674989762593e-07
Epoch : 480 , loss : 0.0

### Prediction using the Unseen data

In [67]:
def predict(sequence):
    model.eval()
    with torch.no_grad():
        # convert the input sequence to the tensor
        source = torch.tensor(sequence).unsqueeze(0)

        hidden,cell = model.encoder(source)
        hidden.squeeze_(0)
        cell.squeeze_(0)
        
        # pass 'SOS' as the first input 
        decoder_input = torch.tensor([SOS])

        # collect the result
        result =[]

        # pass the values one by one to the decoder and genrate the result 
        for _ in range(SEQUENCE_LENGTH + 1):
            output,hidden,cell = model.decoder(decoder_input,hidden,cell)
            # get the real token from the output
            token = output.argmax().item()
            #print(token)

            # Stop adding to the result when EOS is recived
            if token == EOS:
                print(f"final output {result}")
                break
            result.append(token)
            # genrate the decoder_input from the token
            decoder_input = torch.tensor([token])

In [68]:
predict([1,2,3,4,5])

final output [5, 4, 3, 2, 1]


In [79]:
predict([5,8,9,1,1])

final output [1, 1, 8, 5, 7]
